# Muro_MC - Diseño de muro de concreto reforzado (CDMX)

NTC para Diseño y Construcción de Estructuras de Concreto (NTC-Concreto 2023).

Unidades de entrada: kg/cm², cm, ton, ton·m (internamente se trabaja en kgf y cm).

Convención de casos: **Caso 1** = compresión en el extremo derecho (hc1 × bc1); **Caso 2** = compresión en el extremo izquierdo (hc2 × bc2).

Ejecuta las celdas en orden. En la última celda cambia `USAR_EJEMPLO = True` para correr el ejemplo sin que pregunte los datos. Genera `memoria_muro.md` y `esquema_muro.png` en la carpeta del cuaderno.

In [ ]:
import math
import sys
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ----------------------------------------------------------------------------
# CONSTANTES DE DISEÑO (editables)
# ----------------------------------------------------------------------------
EPS_CU = 0.003        # deformación última del concreto
ES = 2.1e6            # kg/cm2, módulo del acero
PHI_F = 0.90          # FR flexión
PHI_V = 0.75          # FR cortante
LAMBDA = 1.0          # concreto de peso normal
RHO_MIN_ALMA = 0.0025 # cuantía mínima de refuerzo distribuido en el alma
RHO_MAX_EXT = 0.06    # cuantía máxima en elementos extremos (criterio de columnas)
RECUB = 3.0           # cm, recubrimiento al estribo (para el núcleo confinado)
COEF_CONF = 0.12      # Ash >= COEF_CONF * s * h'' * f'c/fyh (criterio NTC para columnas dúctiles)
BARRAS = [("#3", 0.71), ("#4", 1.27), ("#5", 1.98), ("#6", 2.85),
          ("#8", 5.07), ("#10", 7.94)]   # área en cm2
DIAM = {"#3": 0.95, "#4": 1.27, "#5": 1.59, "#6": 1.91, "#8": 2.54, "#10": 3.18}

MEM = []


def P(txt=""):
    """Imprime y guarda una línea de la memoria."""
    print(txt)
    MEM.append(txt)


# ----------------------------------------------------------------------------
# PROPIEDADES DE LOS MATERIALES
# ----------------------------------------------------------------------------
def beta1(fc):
    """β1: 0.85 si f'c <= 300 kg/cm2; 1.05 - f'c/1400 >= 0.65 si f'c > 300."""
    return 0.85 if fc <= 300 else max(1.05 - fc / 1400.0, 0.65)


def modulo_elasticidad(fc, clase):
    """Tabla 2.2.1 NTC-Concreto 2023 (agregado grueso calizo), E_c en kg/cm2."""
    if clase == "1A":
        return 14000 * math.sqrt(fc)
    if clase == "1B":
        return 8500 * math.sqrt(fc) + 110000
    return 8000 * math.sqrt(fc)  # Clase 2

In [ ]:
# ----------------------------------------------------------------------------
# GEOMETRÍA
# ----------------------------------------------------------------------------
class Geo:
    def __init__(self, Hm, Lm, tm, hc2, bc2, hc1, bc1):
        self.Hm, self.Lm, self.tm = Hm, Lm, tm
        self.hc2, self.bc2, self.hc1, self.bc1 = hc2, bc2, hc1, bc1
        self.Lw = Lm - hc1 - hc2                      # longitud del alma
        self.A1, self.A2 = hc1 * bc1, hc2 * bc2       # elementos extremos
        self.Aw = self.Lw * tm
        self.Ag = self.A1 + self.A2 + self.Aw
        # centroide de la sección bruta medido desde el borde izquierdo
        self.xg = (self.A2 * hc2 / 2 + self.Aw * (hc2 + self.Lw / 2)
                   + self.A1 * (Lm - hc1 / 2)) / self.Ag


def bloque(a, segs):
    """Área y centroide (desde el borde comprimido) del bloque de profundidad a."""
    A = M1 = y0 = 0.0
    for L_i, b_i in segs:
        if a <= y0:
            break
        l = min(a, y0 + L_i) - y0
        A += b_i * l
        M1 += b_i * l * (y0 + l / 2)
        y0 += L_i
    return A, (M1 / A if A > 0 else 0.0)


def construir_caso(num, comp, g):
    """Datos geométricos del caso medidos desde el borde en compresión."""
    if comp == "der":
        segs = [(g.hc1, g.bc1), (g.Lw, g.tm), (g.hc2, g.bc2)]
        d_c, d_t, yg = g.hc1 / 2, g.Lm - g.hc2 / 2, g.Lm - g.xg
    else:
        segs = [(g.hc2, g.bc2), (g.Lw, g.tm), (g.hc1, g.bc1)]
        d_c, d_t, yg = g.hc2 / 2, g.Lm - g.hc1 / 2, g.xg
    return dict(num=num, comp=comp, segs=segs, d_c=d_c, d_t=d_t, yg=yg)


# ----------------------------------------------------------------------------
# FLEXOCOMPRESIÓN: bloque de esfuerzos + compatibilidad de deformaciones
# ----------------------------------------------------------------------------
def fuerzas(c, caso, As_c, As_t, fc, fy, b1, Lm):
    """Pn (kgf, + compresión) y Mn (kgf*cm, respecto al centroide) para un c dado."""
    a = min(b1 * c, Lm)
    Acon, yc = bloque(a, caso["segs"])
    lista = [(0.85 * fc * Acon, yc)]
    for As, d in ((As_c, caso["d_c"]), (As_t, caso["d_t"])):
        eps = EPS_CU * (c - d) / c                      # + compresión
        f = max(-fy, min(fy, ES * eps))
        F = As * f
        if d <= a:
            F -= 0.85 * fc * As                         # concreto desplazado
        lista.append((F, d))
    Pn = sum(F for F, _ in lista)
    Mn = sum(F * (caso["yg"] - y) for F, y in lista)
    return Pn, Mn, a


def resolver_c(caso, As_c, As_t, Pu, fc, fy, b1, Lm):
    """Bisección sobre c hasta que Pn(c) = Pu/FR (válida con cualquier intervalo)."""
    objetivo = Pu / PHI_F
    lo, hi = 1e-3, 20 * Lm
    Plo = fuerzas(lo, caso, As_c, As_t, fc, fy, b1, Lm)[0]
    Phi = fuerzas(hi, caso, As_c, As_t, fc, fy, b1, Lm)[0]
    if not (Plo <= objetivo <= Phi):
        return None
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if fuerzas(mid, caso, As_c, As_t, fc, fy, b1, Lm)[0] < objetivo:
            lo = mid
        else:
            hi = mid
    c = 0.5 * (lo + hi)
    Pn, Mn, a = fuerzas(c, caso, As_c, As_t, fc, fy, b1, Lm)
    eps_t = EPS_CU * (caso["d_t"] - c) / c
    return dict(c=c, a=a, Pn=Pn, Mn=Mn, phiMn=PHI_F * Mn, eps_t=eps_t)


def proponer_barras(As, nmin=4):
    """Arreglo n x barra con el menor diámetro que dé n <= 12."""
    for nombre, Ab in BARRAS:
        n = max(nmin, math.ceil(As / Ab - 1e-9))
        if n <= 12:
            return n, nombre, n * Ab
    n = math.ceil(As / BARRAS[-1][1])
    return n, BARRAS[-1][0], n * BARRAS[-1][1]


def proponer_malla(rho, t, capas, smax):
    """Barra y separación (múltiplo de 2.5 cm) para la cuantía rho."""
    for nombre, Ab in BARRAS[:4]:
        s = capas * Ab / (rho * t)
        s_ad = min(math.floor(s / 2.5) * 2.5, smax)
        if s_ad >= 10:
            return nombre, s_ad, capas * Ab / (s_ad * t)
    nombre, Ab = BARRAS[3]
    s_ad = 10.0
    return nombre, s_ad, capas * Ab / (s_ad * t)

In [ ]:
# ----------------------------------------------------------------------------
# PROGRAMA PRINCIPAL
# ----------------------------------------------------------------------------
def disenar(d):
    Hm, Lm, tm = d["Hm"], d["Lm"], d["tm"]
    hc2, bc2, hc1, bc1 = d["hc2"], d["bc2"], d["hc1"], d["bc1"]
    fc, fy = d["fc"], d["fy"]
    Pu, Mu, Vu = d["Pu"] * 1000.0, d["Mu"] * 1e5, d["Vu"] * 1000.0   # kgf, kgf*cm, kgf
    g = Geo(Hm, Lm, tm, hc2, bc2, hc1, bc1)
    b1 = beta1(fc)
    Ec = modulo_elasticidad(fc, d["clase"])

    P("# MEMORIA DE CÁLCULO - MURO DE CONCRETO REFORZADO")
    P("NTC para Diseño y Construcción de Estructuras de Concreto, CDMX 2023")
    P()
    P("## 1. Datos de entrada")
    P(f"- Geometría: Hm = {Hm:g} cm, Lm = {Lm:g} cm, tm = {tm:g} cm")
    P(f"- Elemento extremo izquierdo: hc2 = {hc2:g} cm, bc2 = {bc2:g} cm")
    P(f"- Elemento extremo derecho: hc1 = {hc1:g} cm, bc1 = {bc1:g} cm")
    P(f"- Materiales: f'c = {fc:g} kg/cm2, fy = {fy:g} kg/cm2, Es = {ES:.3g} kg/cm2")
    P(f"- Cargas últimas: Pu = {d['Pu']:g} t, Mu = {d['Mu']:g} t*m, Vu = {d['Vu']:g} t")
    P(f"- Factores: FR flexión = {PHI_F}, FR cortante = {PHI_V}, εcu = {EPS_CU}")
    P()
    P("## 2. Propiedades de los materiales y de la sección")
    P(f"- β1 = {b1:.3f}  (β1 = 0.85 si f'c <= 300 kg/cm2; β1 = 1.05 - f'c/1400 >= 0.65 si f'c > 300)")
    P(f"- Ec = {Ec:,.0f} kg/cm2  (Tabla 2.2.1, §2.2.8, concreto clase {d['clase']}, agregado calizo)")
    P(f"- Longitud del alma Lw = Lm - hc1 - hc2 = {g.Lw:g} cm")
    P(f"- Ag = {g.Ag:,.0f} cm2;  centroide desde el borde izquierdo xg = {g.xg:.1f} cm")
    avisos = []
    if g.Lw <= 0:
        raise ValueError("hc1 + hc2 debe ser menor que Lm")

    # --- verificaciones geométricas generales -----------------------------
    P()
    P("## 3. Verificaciones geométricas (NTC-Concreto 2023, cap. 6.5, §6.5.3)")
    lim = 0.3 * fc * g.Ag
    if Pu <= lim:
        P(f"- Pu = {Pu/1000:.1f} t <= 0.3 f'c Ag = {lim/1000:.1f} t: el muro se diseña a flexión como viga en voladizo; L/t <= 70.")
        Lt_max = 70
    else:
        P(f"- Pu = {Pu/1000:.1f} t > 0.3 f'c Ag = {lim/1000:.1f} t: trabaja a flexocompresión; L/t <= 40.")
        avisos.append("Pu > 0.3 f'c Ag: revisar FR de flexocompresión (se usó FR = 0.90 por indicación).")
        Lt_max = 40
    ok = Lm / tm <= Lt_max
    P(f"- Lm/tm = {Lm/tm:.1f} {'<=' if ok else '>'} {Lt_max} -> {'CUMPLE' if ok else 'NO CUMPLE'}")
    if not ok:
        avisos.append("Lm/tm excede el límite de esbeltez.")
    ok = tm >= 14
    P(f"- tm = {tm:g} cm {'>=' if ok else '<'} 14 cm (espesor mínimo muros de carga) -> {'CUMPLE' if ok else 'NO CUMPLE'}")
    r = Hm / Lm
    b_borde = (0.25 - 0.1 * r) * Lm
    b_borde = min(b_borde, 0.4 * Hm) if r <= 1.2 else 0.15 * Lm
    P(f"- Hm/Lm = {r:.2f}: longitud mínima de concentración del refuerzo de borde = {b_borde:.1f} cm")
    for nom, h in (("hc1", hc1), ("hc2", hc2)):
        ok = h >= b_borde
        P(f"  {nom} = {h:g} cm {'>=' if ok else '<'} {b_borde:.1f} cm -> {'CUMPLE' if ok else 'REVISAR'}")
        if not ok:
            avisos.append(f"{nom} menor que la longitud de borde sugerida.")

    # --- FLEXOCOMPRESIÓN, PASO 1: método T y C -----------------------------
    casos = [construir_caso(1, "der", g), construir_caso(2, "izq", g)]
    P()
    P("## 4. Flexocompresión - Paso 1: método T y C (acero concentrado al centro de los elementos extremos)")
    P("Equilibrio de Pu y Mu respecto al centroide de la sección, con las fuerzas en los centroides de los extremos:")
    P("  Fc = (Mu + Pu*at)/(ac + at);  Ft = Pu - Fc   (+ compresión, - tensión)")
    P("  Elemento en tensión:     As = T/(FR fy)")
    P("  Elemento en compresión:  As = (C/FR - 0.85 f'c Ae)/(fy - 0.85 f'c)")
    req = {"izq": 0.0, "der": 0.0}
    for cs in casos:
        ac, at = cs["yg"] - cs["d_c"], cs["d_t"] - cs["yg"]
        Fc = (Mu + Pu * at) / (ac + at)
        Ft = Pu - Fc
        e_c, e_t = ("der", "izq") if cs["comp"] == "der" else ("izq", "der")
        Ae = {"der": g.A1, "izq": g.A2}
        out = {}
        for key, F in ((e_c, Fc), (e_t, Ft)):
            if F < 0:
                out[key] = -F / (PHI_F * fy)
            else:
                out[key] = max(0.0, (F / PHI_F - 0.85 * fc * Ae[key]) / (fy - 0.85 * fc))
            req[key] = max(req[key], out[key])
        cs.update(Fc=Fc, Ft=Ft, e_c=e_c, e_t=e_t, req=out)
        P(f"- Caso {cs['num']} (compresión en extremo {'derecho' if cs['comp']=='der' else 'izquierdo'}): "
          f"ac = {ac:.1f} cm, at = {at:.1f} cm")
        P(f"    Fc = {Fc/1000:.1f} t (compresión), Ft = {Ft/1000:.1f} t ({'tensión' if Ft<0 else 'compresión'})")
        P(f"    As requerida: extremo {e_c} = {out[e_c]:.2f} cm2;  extremo {e_t} = {out[e_t]:.2f} cm2")

    # cuantías límite en elementos extremos
    As_izq = max(req["izq"], 20.0 / fy * g.A2)
    As_der = max(req["der"], 20.0 / fy * g.A1)
    P()
    P(f"- Envolvente de ambos casos y cuantía mínima 20/fy = {20/fy:.4f}: "
      f"As_izq = {As_izq:.2f} cm2, As_der = {As_der:.2f} cm2")

    # --- PASO 2: bisección sobre c --------------------------------------------
    P()
    P("## 5. Flexocompresión - Paso 2: bloque de esfuerzos, εcu = 0.003 y profundidad del eje neutro c")
    P("Con a = β1 c, concreto 0.85 f'c sobre el área comprimida (alma + elemento extremo), acero en el centro de los")
    P("elementos extremos con fs = Es*εs <= fy. Se resuelve c por bisección de  Pn(c) = Pu/FR  y se verifica FR*Mn >= Mu.")
    k = 1.0
    res = None
    for it in range(600):
        res = []
        todos_ok = True
        for cs in casos:
            A_c = (As_der if cs["comp"] == "der" else As_izq) * k
            A_t = (As_izq if cs["comp"] == "der" else As_der) * k
            r_ = resolver_c(cs, A_c, A_t, Pu, fc, fy, b1, Lm)
            if r_ is None or r_["phiMn"] < Mu:
                todos_ok = False
            res.append((A_c, A_t, r_))
        if todos_ok:
            break
        k *= 1.02
    if not todos_ok:
        avisos.append("No se logró cumplir FR*Mn >= Mu: aumentar dimensiones o f'c.")
    As_izq_f, As_der_f = As_izq * k, As_der * k
    if k > 1:
        P(f"- La verificación con compatibilidad de deformaciones obligó a aumentar el acero {100*(k-1):.0f}% sobre el de T y C.")
    for cs, (A_c, A_t, r_) in zip(casos, res):
        cs.update(A_c=A_c, A_t=A_t, r=r_)
        P(f"- Caso {cs['num']}: As comp = {A_c:.2f} cm2, As tens = {A_t:.2f} cm2")
        if r_ is None:
            P("    Sin solución de equilibrio para Pu.")
            continue
        P(f"    c = {r_['c']:.2f} cm,  a = β1 c = {r_['a']:.2f} cm,  εt = {r_['eps_t']:.4f}")
        P(f"    Pn = {r_['Pn']/1000:.1f} t (Pu/FR = {Pu/PHI_F/1000:.1f} t),  FR*Mn = {r_['phiMn']/1e5:.1f} t*m "
          f"{'>=' if r_['phiMn']>=Mu else '<'} Mu = {Mu/1e5:.1f} t*m -> {'CUMPLE' if r_['phiMn']>=Mu else 'NO CUMPLE'}")
        if r_["eps_t"] < 0.005:
            avisos.append(f"Caso {cs['num']}: εt = {r_['eps_t']:.4f} < 0.005, la sección no es controlada por tensión; revisar FR = 0.90.")

    # arreglos de barras y confinamiento
    P()
    P("## 6. Acero vertical y transversal en los elementos extremos")
    ext = {}
    for nom, As, A, h, b in (("izq", As_izq_f, g.A2, hc2, bc2), ("der", As_der_f, g.A1, hc1, bc1)):
        n, bar, As_p = proponer_barras(As)
        rho = As_p / A
        if rho > RHO_MAX_EXT:
            avisos.append(f"Extremo {nom}: ρ = {rho:.3f} > {RHO_MAX_EXT}; aumentar el elemento extremo.")
        hx, hy = b - 2 * RECUB, h - 2 * RECUB            # núcleo: espesor y longitud
        Ac = hx * hy
        coef = max(0.3 * (A / Ac - 1) * fc / fy, COEF_CONF * fc / fy)
        ash_x = coef * hx        # cm2/cm, ramas paralelas a la longitud del muro
        ash_y = coef * hy        # cm2/cm, ramas paralelas al espesor
        smax = min(8 * DIAM[bar], 24 * DIAM["#3"], tm / 2)
        ext[nom] = dict(As=As, n=n, bar=bar, As_p=As_p, rho=rho, ash_x=ash_x, ash_y=ash_y, smax=smax)
        P(f"- Extremo {'izquierdo' if nom=='izq' else 'derecho'} ({h:g} x {b:g} cm): As requerida = {As:.2f} cm2 -> "
          f"{n} {bar} (As = {As_p:.2f} cm2, ρ = {rho:.4f})")
        P(f"    Ash/s (ramas // longitud) = {ash_x:.4f} cm2/cm;  Ash/s (ramas // espesor) = {ash_y:.4f} cm2/cm")
        P(f"    Separación máx. contra pandeo: min(8 db = {8*DIAM[bar]:.1f}, 24 db_est = {24*DIAM['#3']:.1f}, tm/2 = {tm/2:.1f}) = {smax:.1f} cm")
    P("  (Ash/s >= max[0.3 (Ag/Ac - 1) f'c/fyh ; 0.12 f'c/fyh] * h'',  h'' = dimensión del núcleo c-a-c de estribos)")

    # --- CORTANTE -------------------------------------------------------------
    P()
    P("## 7. Revisión por cortante (NTC-Concreto 2023, cap. 6.5/7.5/8.5)")
    Acv = tm * Lm
    r = Hm / Lm
    if r <= 1.5:
        ac_ = 0.80
    elif r >= 2.0:
        ac_ = 0.53
    else:
        ac_ = 0.80 + (r - 1.5) / 0.5 * (0.53 - 0.80)
    if Pu < 0:
        ac_ = max(0.0, 0.53 * (1 + Pu / (35 * g.Ag)))
    Vc = PHI_V * ac_ * LAMBDA * math.sqrt(fc) * Acv
    Vmax = 2.0 * PHI_V * math.sqrt(fc) * Acv
    P(f"- Acv = tm*Lm = {Acv:,.0f} cm2;  Hm/Lm = {r:.2f} -> αc = {ac_:.3f}")
    P(f"- VcR = FR αc λ √f'c Acv = {Vc/1000:.1f} t")
    P(f"- Límite por aplastamiento del puntal: Vu <= 2 FR √f'c Acv = {Vmax/1000:.1f} t -> "
      f"{'CUMPLE' if Vu <= Vmax else 'NO CUMPLE (aumentar tm o f'+chr(39)+'c)'}")
    if Vu > Vmax:
        avisos.append("Vu excede el límite de aplastamiento del puntal.")
    necesita = (Vu / PHI_V - ac_ * LAMBDA * math.sqrt(fc) * Acv) / (fy * Acv)
    rho_h = max(necesita, RHO_MIN_ALMA)
    rho_v = max(RHO_MIN_ALMA, 0.0025 + 0.5 * (2.5 - r) * (rho_h - 0.0025))
    VR = PHI_V * (ac_ * LAMBDA * math.sqrt(fc) + rho_h * fy) * Acv
    P(f"- ρh requerida = (Vu/FR - αc λ √f'c Acv)/(fy Acv) = {necesita:.5f};  ρh adoptada = max(·, 0.0025) = {rho_h:.5f}")
    P(f"- ρv >= 0.0025 + 0.5 (2.5 - Hm/Lm)(ρh - 0.0025) = {rho_v:.5f}")
    P(f"- VR = FR (αc λ √f'c + ρh fy) Acv = {VR/1000:.1f} t {'>=' if VR >= Vu else '<'} Vu = {Vu/1000:.1f} t -> "
      f"{'CUMPLE' if VR >= Vu else 'NO CUMPLE'}")
    capas = 2 if tm > 15 else 1
    sv_max = min(3 * tm, 45, Lm / 3)
    sh_max = min(3 * tm, 45, Lm / 5)
    bv, sv, rv_real = proponer_malla(rho_v, tm, capas, sv_max)
    bh, sh, rh_real = proponer_malla(rho_h, tm, capas, sh_max)
    Asv_tot = rv_real * tm * g.Lw
    Ash_cm = rh_real * tm
    P(f"- Malla en el alma ({capas} capa(s)): vertical {bv} @ {sv:g} cm (ρv = {rv_real:.5f}, s máx = {sv_max:.0f} cm); "
      f"horizontal {bh} @ {sh:g} cm (ρh = {rh_real:.5f}, s máx = {sh_max:.0f} cm)")
    P(f"- Acero vertical en el alma: {Asv_tot:.2f} cm2 totales en Lw = {g.Lw:g} cm ({rv_real*tm*100:.2f} cm2/m)")
    P(f"- Acero horizontal en el alma: {Ash_cm:.4f} cm2/cm = {Ash_cm*100:.2f} cm2/m de altura "
      f"({Ash_cm*Hm:.1f} cm2 en Hm = {Hm:g} cm)")

    # --- RESUMEN ---------------------------------------------------------------
    P()
    P("## 8. Resumen de resultados")
    for cs in casos:
        r_ = cs["r"]
        if r_ is None:
            continue
        P(f"- Caso {cs['num']} (compresión {'derecha' if cs['comp']=='der' else 'izquierda'}): c = {r_['c']:.2f} cm; "
          f"εt = {r_['eps_t']:.4f}; FR*Mn = {r_['phiMn']/1e5:.1f} t*m")
    for nom, tit in (("izq", "Extremo izquierdo"), ("der", "Extremo derecho")):
        e = ext[nom]
        P(f"- {tit}: As vertical = {e['As']:.2f} cm2 ({e['n']} {e['bar']}); Ash/s = {e['ash_x']:.4f} / {e['ash_y']:.4f} cm2/cm")
    P(f"- Alma: As vertical = {Asv_tot:.2f} cm2 ({bv} @ {sv:g}); As horizontal = {Ash_cm*100:.2f} cm2/m ({bh} @ {sh:g})")
    if avisos:
        P()
        P("### Avisos")
        for a in avisos:
            P(f"- {a}")
    P()
    P("### Notas")
    P("- Las fórmulas se basan en las imágenes aportadas (β1, Tabla 2.2.1) y en el resumen de la NTC-Concreto 2023 de")
    P("  la tesis BUAP 2025 (muros). Confirmar numeración de apartados, cuantías mínimas del cap. 6.5/7.5/8.5, el límite")
    P("  de aplastamiento del puntal y el coeficiente de confinamiento contra el texto oficial antes de usar en proyecto.")
    P("- El acero distribuido del alma no se consideró en la flexión (criterio conservador).")
    return g, casos, ext, dict(Asv=Asv_tot, Ash=Ash_cm, bv=bv, sv=sv, bh=bh, sh=sh, rho_v=rv_real, rho_h=rh_real)

In [ ]:
# ----------------------------------------------------------------------------
# ESQUEMA
# ----------------------------------------------------------------------------
def dibujar(g, casos, ext, alma, ruta):
    fig, axes = plt.subplots(2, 1, figsize=(12, 9.5))
    bmax = max(g.bc1, g.bc2, g.tm)
    for ax, cs in zip(axes, casos):
        r = cs["r"]
        ax.add_patch(Rectangle((0, -g.bc2 / 2), g.hc2, g.bc2, fc="#e6e6e6", ec="k", lw=1.5))
        ax.add_patch(Rectangle((g.hc2, -g.tm / 2), g.Lw, g.tm, fc="#e6e6e6", ec="k", lw=1.5))
        ax.add_patch(Rectangle((g.Lm - g.hc1, -g.bc1 / 2), g.hc1, g.bc1, fc="#e6e6e6", ec="k", lw=1.5))
        x_izq, x_der = g.hc2 / 2, g.Lm - g.hc1 / 2
        if r:
            a = r["a"]
            tramos = [(0, g.hc2, g.bc2), (g.hc2, g.Lm - g.hc1, g.tm), (g.Lm - g.hc1, g.Lm, g.bc1)]
            for x1, x2, b in tramos:
                if cs["comp"] == "der":
                    lo, hi = max(x1, g.Lm - a), x2
                else:
                    lo, hi = x1, min(x2, a)
                if hi > lo:
                    ax.add_patch(Rectangle((lo, -b / 2), hi - lo, b, fc="#9ecae1", ec="none", hatch="///", alpha=0.8))
            xe = g.Lm - r["c"] if cs["comp"] == "der" else r["c"]
            ax.plot([xe, xe], [-bmax * 0.7, bmax * 0.75], color="r", ls="--", lw=1.8)
            ax.text(xe, bmax * 0.8, f"eje neutro, c = {r['c']:.1f} cm", color="r", ha="center", va="bottom", fontsize=10)
        for x, nom in ((x_izq, "izq"), (x_der, "der")):
            ax.plot(x, 0, "ko", ms=9)
            e = ext[nom]
            ax.text(x, -bmax * 0.95, f"As,v = {e['As']:.1f} cm²\n({e['n']} {e['bar']})\nAsh/s = {max(e['ash_x'], e['ash_y']):.3f} cm²/cm",
                    ha="center", va="top", fontsize=9)
        ax.text(g.Lm / 2, -g.tm / 2 - bmax * 0.15,
                f"Alma: As,v = {alma['Asv']:.1f} cm² ({alma['bv']} @ {alma['sv']:g})   "
                f"As,h = {alma['Ash']*100:.2f} cm²/m ({alma['bh']} @ {alma['sh']:g})",
                ha="center", va="top", fontsize=9)
        ax.annotate("", xy=(0, bmax * 1.9), xytext=(g.Lm, bmax * 1.9), arrowprops=dict(arrowstyle="<->"))
        ax.text(g.Lm / 2, bmax * 1.95, f"Lm = {g.Lm:g} cm", ha="center", fontsize=9)
        lado = "derecho" if cs["comp"] == "der" else "izquierdo"
        ax.set_title(f"Caso {cs['num']}: compresión en el extremo {lado}", fontsize=12)
        ax.set_xlim(-0.04 * g.Lm, 1.04 * g.Lm)
        ax.set_ylim(-bmax * 3.2, bmax * 2.6)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(ruta, dpi=150)
    plt.show()

In [ ]:
# ----------------------------------------------------------------------------
# ENTRADA DE DATOS
# ----------------------------------------------------------------------------
EJEMPLO = dict(Hm=1200.0, Lm=600.0, tm=20.0, hc2=100.0, bc2=40.0, hc1=100.0, bc1=40.0,
               fc=250.0, fy=4200.0, Pu=175.5, Mu=1527.3, Vu=150.0, clase="1A")

CAMPOS = [("Hm", "Hm, altura del muro (cm)"), ("Lm", "Lm, longitud total (cm)"), ("tm", "tm, espesor (cm)"),
          ("hc2", "hc2, longitud del elemento extremo izquierdo (cm)"),
          ("bc2", "bc2, espesor del elemento extremo izquierdo (cm)"),
          ("hc1", "hc1, longitud del elemento extremo derecho (cm)"),
          ("bc1", "bc1, espesor del elemento extremo derecho (cm)"),
          ("fc", "f'c (kg/cm2)"), ("fy", "fy (kg/cm2)"),
          ("Pu", "Pu, carga axial última (ton, + compresión)"),
          ("Mu", "Mu, momento último (ton*m)"), ("Vu", "Vu, cortante último (ton)")]


def pedir_datos():
    print("Datos de entrada (Enter acepta el valor entre corchetes)\n")
    d = {}
    for k, txt in CAMPOS:
        s = input(f"{txt} [{EJEMPLO[k]:g}]: ").strip()
        d[k] = float(s) if s else EJEMPLO[k]
    s = input("Clase de concreto Tabla 2.2.1 (1A / 1B / 2) [1A]: ").strip().upper()
    d["clase"] = s if s in ("1A", "1B", "2") else "1A"
    return d

In [ ]:
# Cambia a True para correr el ejemplo sin que pregunte los datos
USAR_EJEMPLO = False

MEM.clear()
datos = dict(EJEMPLO) if USAR_EJEMPLO else pedir_datos()
g, casos, ext, alma = disenar(datos)
dibujar(g, casos, ext, alma, "esquema_muro.png")
with open("memoria_muro.md", "w", encoding="utf-8") as f:
    f.write("\n".join(MEM))
print("\nArchivos generados: memoria_muro.md, esquema_muro.png")